<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI for Algorithmic Trading
## Chapter 7 · Baseline Technical Strategies and Vectorised Backtesting

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.1<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


### Notebook Goals

This notebook walks through the SMA crossover example from Chapter 7.

- Load SPY prices from the static EOD panel.
- Build buy-and-hold and 20/100-day SMA positions.
- Run vectorised backtests and compare equity curves and metrics.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8")  # set notebook-wide plotting style
try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
    set_matplotlib_formats("svg")
except Exception:
    pass
plt.rcParams.update({"figure.dpi": 300})

PROJECT_ROOT = Path("..").resolve()  # project root relative to notebooks
CODE_DIR = PROJECT_ROOT / "code"  # folder with chapter scripts
if str(CODE_DIR) not in sys.path:  # ensure scripts are importable
    sys.path.insert(0, str(CODE_DIR))

import ch05_eod_engineering as ch05  # EOD data helpers
import ch07_baseline_strategies as ch07  # SMA and backtest helpers


### 1. Positions for Buy-and-Hold and SMA

We first build the benchmark and SMA position series using the same helpers as the standalone script.

In [ ]:
panel = ch05.load_eod_panel()
prices_spy = panel["SPY"].astype(float)

pos_bh = ch07.build_positions_buy_and_hold(prices_spy)
pos_sma = ch07.build_positions_sma(prices_spy)

pos_bh.head(), pos_sma.head()


### 2. Vectorised Backtest and Metrics

The `backtest_strategy` helper applies transaction costs and financing to obtain daily net returns, which we then summarise with `compute_metrics`.

In [ ]:
net_bh = ch07.backtest_strategy(prices_spy, pos_bh)
net_sma = ch07.backtest_strategy(prices_spy, pos_sma)

metrics_bh = ch07.compute_metrics(net_bh)
metrics_sma = ch07.compute_metrics(net_sma)

print("Buy-and-hold metrics:")
for key, value in metrics_bh.items():
    print(f"  {key:15s} : {value: .4f}")

print("\nSMA 20/100 metrics:")
for key, value in metrics_sma.items():
    print(f"  {key:15s} : {value: .4f}")


### 3. Equity-Curve Comparison

We mirror the figure from the script and keep it available for quick experimentation with different parameters.

In [ ]:
wealth_bh = (1.0 + net_bh).cumprod()
wealth_sma = (1.0 + net_sma).cumprod()

fig, ax = plt.subplots(figsize=(7.0, 3.8))
ax.plot(wealth_bh.index, wealth_bh.values, label="buy-and-hold (SPY)")
ax.plot(wealth_sma.index, wealth_sma.values, label="20/100-day SMA")
ax.set_ylabel("wealth (normalised)")
ax.set_xlabel("date")
ax.set_title("SPY: SMA crossover vs buy-and-hold")
ax.legend(loc="upper left")
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()


### Takeaways

- Vectorised positions and cost-aware backtests give quick feedback on baseline strategies.
- Simple SMAs already display rich behaviour once costs, financing, and drawdowns are taken into account.
- Later chapters reuse the same helpers when comparing more advanced models against these baselines.


<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">